# Training Source 1: EDA and Iterative Cleaning

Source 1 is the deduplicated reference source.

## Findings from raw data

2,206,821 records: 1,323,633 US and 883,188 India. Names and addresses are populated in every record.

## Cleaning decision

The observed missing addresses in this source are retained as empty fields: these records can still match by name and country. Cleaning trims whitespace at field boundaries and preserves spelling, case, punctuation, script, and every row. Raw input remains unchanged.

The workflow is split into three visible iterations: raw profiling, conservative cleaning to a compressed TSV, then post-cleaning QA. It streams data instead of loading the large source file into memory. Run cells from top to bottom. The generated file is `dataset/cleaned/train/train_source1.cleaned.tsv.gz`.

## EDA results from the run

- Name lengths: 69,652 are 1–10 characters; 1,671,332 are 11–30; 465,576 are 31–60; 261 are 61+.
- Address lengths: 335,046 are 11–30 characters; 1,169,310 are 31–60; 702,465 are 61+.
- Cleaning QA passed: all 2,206,821 rows and all four columns were preserved; no blank names or addresses were introduced, and no boundary whitespace remains in name/address fields.


In [ ]:
import csv
import gzip
from collections import Counter
from pathlib import Path

filename = 'train_source1.tsv'
search_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next((p for p in search_roots if (p / "dataset" / "train" / filename).is_file()), None)
if project_root is None:
    raise FileNotFoundError(f"Could not locate dataset/train/{filename} from {Path.cwd()}")
raw_path = project_root / "dataset" / "train" / filename
clean_dir = project_root / "dataset" / "cleaned" / "train"
clean_dir.mkdir(parents=True, exist_ok=True)
clean_path = clean_dir / (raw_path.stem + ".cleaned.tsv.gz")
print("Raw file:", raw_path)
print("Clean output:", clean_path)


In [ ]:
# Iteration 1: profile the raw TSV. Length buckets give a compact view of field variation.
row_count = 0
countries = Counter()
blank_counts = Counter()
name_lengths = Counter()
address_lengths = Counter()
samples = []

def length_bucket(n):
    if n == 0: return "0"
    if n <= 10: return "1-10"
    if n <= 30: return "11-30"
    if n <= 60: return "31-60"
    return "61+"

with raw_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    columns = reader.fieldnames
    for row in reader:
        row_count += 1
        countries[(row.get("country") or "").strip()] += 1
        name = row.get("business_name") or ""
        address = row.get("business_address") or ""
        blank_counts["business_name"] += not bool(name.strip())
        blank_counts["business_address"] += not bool(address.strip())
        name_lengths[length_bucket(len(name.strip()))] += 1
        address_lengths[length_bucket(len(address.strip()))] += 1
        if len(samples) < 5:
            samples.append(row)

print("Columns:", columns)
print("Rows:", f"{row_count:,}")
print("Country counts:", dict(countries))
print("Blank values:", dict(blank_counts))
print("Business name length buckets:", dict(name_lengths))
print("Address length buckets:", dict(address_lengths))
print("Sample records:")
for sample in samples: print(sample)


In [ ]:
# Iteration 2: conservative cleaning. Trim only field boundaries and keep blanks blank.
# Do not lowercase, transliterate, remove punctuation, or discard incomplete-address rows;
# those changes could erase useful entity-resolution evidence.
clean_rows = 0
clean_blank_counts = Counter()
with raw_path.open("r", encoding="utf-8", newline="") as src, gzip.open(clean_path, "wt", encoding="utf-8", newline="") as dst:
    reader = csv.DictReader(src, delimiter="\t")
    writer = csv.DictWriter(dst, fieldnames=reader.fieldnames, delimiter="\t", lineterminator="\n")
    writer.writeheader()
    for row in reader:
        cleaned = {key: (value.strip() if value is not None else "") for key, value in row.items()}
        for field in ("business_name", "business_address"):
            clean_blank_counts[field] += not bool(cleaned[field])
        writer.writerow(cleaned)
        clean_rows += 1
print("Cleaned rows written:", f"{clean_rows:,}")
print("Blank values after cleaning:", dict(clean_blank_counts))


In [ ]:
# Iteration 3: verify output schema, row preservation, and whitespace cleanup.
qa_rows = 0
qa_blank_counts = Counter()
qa_whitespace_issues = 0
with gzip.open(clean_path, "rt", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    output_columns = reader.fieldnames
    for row in reader:
        qa_rows += 1
        for field in ("business_name", "business_address"):
            value = row.get(field) or ""
            qa_blank_counts[field] += not bool(value)
            qa_whitespace_issues += value != value.strip()
assert qa_rows == row_count, f"Row count changed: raw={{row_count}}, clean={{qa_rows}}"
assert output_columns == columns, f"Column order changed: {{output_columns}}"
assert qa_whitespace_issues == 0
print("Post-clean QA: PASS")
print("Rows preserved:", f"{qa_rows:,}")
print("Columns preserved:", output_columns)
print("Blank fields remain (intentionally):", dict(qa_blank_counts))
